In [ ]:

%load_ext ElasticNotebook

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten_cpu/q20_small_rewrite/checkpoints/post_cell_19_try_0.pickle

In [ ]:
%%PandasProfile
### cell 20 ###

def add_gap_rows_2(essay):
    cols_to_keep = ['discourse_start', 'discourse_end', 'discourse_type', 'gap_length', 'gap_end_length']
    df_essay = train.query(f"id == '{essay}'")[cols_to_keep].reset_index(drop=True)

    # Defining a DataFrame to store additional gap rows
    gaps = []

    prev_end = 0
    for i_2 in range(len(df_essay)):
        if i_2 > 0 and df_essay.at[i_2, "gap_length"] > 0:
            start = df_essay.at[i_2-1, "discourse_end"] + 1
            end = df_essay.at[i_2, 'discourse_start'] - 1
            gaps.append([start, end, "Nothing", np.nan, np.nan])

    # Adding a gap at the end if required
    last_row = df_essay.iloc[-1]
    if last_row['gap_end_length'] > 0:
        start = last_row['discourse_end'] + 1
        end = start + last_row['gap_end_length'] - 1
        gaps.append([start, end, "Nothing", np.nan, np.nan])

    # Creating a DataFrame for gaps and concatenating it with the original one
    if gaps:
        gaps_df = pd.DataFrame(gaps, columns=cols_to_keep)
        df_essay = pd.concat([df_essay, gaps_df], ignore_index=True)

    # Sorting and returning the result
    return df_essay.sort_values(by='discourse_start').reset_index(drop=True)


def print_colored_essay(essay):
    df_essay = add_gap_rows_2(essay)
    essay_file = f"/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/input/feedback-prize-2021/train/{essay}.txt"

    # Extracting entities from the dataframe
    ents = [
        {'start': int(row['discourse_start']), 'end': int(row['discourse_end']), 'label': row['discourse_type']}
        for _, row in df_essay.iterrows()
    ]

    # Reading the text content
    with open(essay_file, 'r') as file:
        data = file.read()

    doc2 = {
        "text": data,
        "ents": ents,
    }

    # Entity colors
    colors = {'Lead': '#EE11D0','Position': '#AB4DE1','Claim': '#1EDE71','Evidence': '#33FAFA','Counterclaim': '#4253C1','Concluding Statement': 'yellow','Rebuttal': 'red'}
    options = {"ents": df_essay.discourse_type.unique().tolist(), "colors": colors}

print_colored_essay(IREWR_plug_2)